# Tool-Augmented LLM App (LangChain + OpenRouter)
**Kunal Ahuja** | Automate with GenAI & Agents | Module 4 Graded Assignment

Pipeline: user question -> LLM decides (answer directly OR call a tool) -> tool runs -> result goes back to the LLM -> final answer.
- **Tools:** `get_weather(city, unit)` (Open-Meteo, no API key) and `calculator(expression)` (safe AST evaluator).
- **Memory / caching:** LangChain `InMemoryCache` for LLM calls plus a small dictionary cache for weather lookups.
- **Comparison:** vanilla LLM (no tools, no cache) vs tool-augmented LLM (tools + cache).

The API key is read from Colab Secrets (`OPENROUTER_API_KEY`) or a hidden prompt and is never stored in the notebook.

## Cell 1: Install, imports and secure API key

In [1]:
%pip install -q langchain langchain-openai langchain-core requests pandas

import ast
import operator
import time
from getpass import getpass

import pandas as pd
import requests
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_API_KEY")   # Colab Secrets (key icon)
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Enter your OpenRouter API key: ")

api_key = (api_key or "").strip()            # drop stray spaces/newlines from pasting
if not api_key:
    raise ValueError("No API key found. Add a Colab secret named OPENROUTER_API_KEY "
                     "(Notebook access ON) and re-run this cell.")
print(f"API key loaded ({len(api_key)} characters, "
      f"{'looks like an OpenRouter key' if api_key.startswith('sk-or-') else 'does NOT start with sk-or-, check it'})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 3.5 MB/s eta 0:00:00
Enter your OpenRouter API key: ··········
API key loaded (73 characters, looks like an OpenRouter key)


## Cell 2: Models (one OpenRouter endpoint, two configurations)

In [2]:
LLM_MODEL = "openai/gpt-4o-mini"   # change the slug here to try another model

def make_llm(use_cache: bool) -> ChatOpenAI:
    """Build an OpenRouter-backed chat model.
    cache=False disables LangChain caching for this instance (used for the
    vanilla baseline); cache=None follows the global cache set in Cell 3."""
    return ChatOpenAI(
        model=LLM_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        temperature=0,                       # deterministic, so caching is meaningful
        cache=None if use_cache else False,
    )

vanilla_llm = make_llm(use_cache=False)      # baseline: no tools, no cache
agent_llm = make_llm(use_cache=True)         # used by the agent below

## Cell 3: Memory / caching

In [3]:
# Global in-memory cache: an identical request (same messages + parameters)
# is answered from memory instead of calling the API again.
set_llm_cache(InMemoryCache())

# Tool-level cache: repeated weather lookups for the same city/unit reuse the
# earlier result instead of calling the weather API again.
weather_cache = {}
stats = {"weather_api_calls": 0, "weather_cache_hits": 0}

## Cell 4: Tools

In [4]:
WEATHER_CODES = {0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
                 45: "fog", 48: "freezing fog", 51: "light drizzle", 53: "drizzle",
                 55: "heavy drizzle", 61: "light rain", 63: "rain", 65: "heavy rain",
                 71: "light snow", 73: "snow", 75: "heavy snow", 80: "rain showers",
                 81: "heavy rain showers", 95: "thunderstorm"}

@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """Get the current weather for a city. `unit` is 'celsius' or 'fahrenheit'."""
    unit = unit.lower().strip()
    if unit not in ("celsius", "fahrenheit"):
        unit = "celsius"
    key = (city.lower().strip(), unit)
    if key in weather_cache:                          # tool-level cache hit
        stats["weather_cache_hits"] += 1
        return weather_cache[key]                     # identical text, so the LLM cache also hits
    try:
        # Step 1: city name -> latitude/longitude (Open-Meteo geocoding, no key)
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city, "count": 1}, timeout=10).json()
        if not geo.get("results"):
            return f"Could not find a city named '{city}'."
        place = geo["results"][0]
        # Step 2: coordinates -> current conditions
        w = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude": place["latitude"], "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            "temperature_unit": unit}, timeout=10).json()["current"]
        stats["weather_api_calls"] += 1
        symbol = "C" if unit == "celsius" else "F"
        result = (f"{place['name']}, {place.get('country', '')}: {w['temperature_2m']}°{symbol}, "
                  f"{WEATHER_CODES.get(w['weather_code'], 'unknown conditions')}, "
                  f"humidity {w['relative_humidity_2m']}%, wind {w['wind_speed_10m']} km/h")
        weather_cache[key] = result
        return result
    except Exception as e:                            # network/API failure
        return f"Weather lookup failed: {e}"

# Safe calculator: parses the expression into an AST and only allows basic
# arithmetic nodes, so it never uses eval() on untrusted text.
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
        ast.Mod: operator.mod}

def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("Unsupported expression")

@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as '42 * 19' or '(3 + 4) / 2'."""
    try:
        return str(_eval(ast.parse(expression.replace("×", "*").replace("x", "*"),
                                   mode="eval").body))
    except Exception as e:
        return f"Calculator error: {e}"

TOOLS = {t.name: t for t in (get_weather, calculator)}
print("Tools available:", list(TOOLS))

Tools available: ['get_weather', 'calculator']


## Cell 5: Agent workflow (the model decides: answer directly or call a tool)

In [5]:
SYSTEM_PROMPT = ("You are a helpful assistant. Use the get_weather tool for current "
                 "weather and the calculator tool for arithmetic. For general knowledge "
                 "questions, answer directly without tools.")

# Expose the tool schemas to the model (structured function calling).
agent_with_tools = agent_llm.bind_tools(list(TOOLS.values()))

def run_agent(question: str, max_steps: int = 5, verbose: bool = True) -> dict:
    """Loop: ask the LLM -> if it requests tools, run them and send the results
    back -> repeat until it answers in plain text."""
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]
    start, tools_used, tokens = time.time(), [], 0
    for _ in range(max_steps):
        ai = agent_with_tools.invoke(messages)
        # Keep only content + tool calls in the history. The raw reply carries a
        # unique run id, which would change the prompt on every run and make the
        # next step miss the cache.
        messages.append(AIMessage(content=ai.content, tool_calls=ai.tool_calls))
        tokens += (ai.usage_metadata or {}).get("total_tokens", 0)
        if not ai.tool_calls:                         # model chose to answer directly
            break
        for call in ai.tool_calls:                    # model chose to use tool(s)
            result = TOOLS[call["name"]].invoke(call["args"])
            tools_used.append(f"{call['name']}({call['args']})")
            if verbose:
                print(f"   -> tool call: {call['name']}({call['args']}) = {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return {"answer": ai.content, "tools_used": tools_used,
            "latency_sec": round(time.time() - start, 2), "total_tokens": tokens}

def vanilla(question: str) -> dict:
    """Baseline: plain LLM call, no tools, no cache."""
    start = time.time()
    ai = vanilla_llm.invoke([HumanMessage(question)])
    return {"answer": ai.content, "latency_sec": round(time.time() - start, 2),
            "total_tokens": (ai.usage_metadata or {}).get("total_tokens", 0)}

## Cell 6: Test queries - vanilla vs tool-augmented (and cached repeat)

In [6]:
QUESTIONS = [
    "What's the weather in Paris right now? Give the temperature in celsius.",
    "What's 42 x 19?",
    "What's the weather in Paris and what is 42 x 19? Summarize both results in one sentence.",
    "What is the capital of France?",   # no tool needed: tests answering directly
]

rows = []
for q in QUESTIONS:
    print(f"\n{'='*70}\nQUESTION: {q}\n{'='*70}")
    v = vanilla(q)
    print(f"\n--- VANILLA LLM ({v['latency_sec']}s, {v['total_tokens']} tokens) ---\n{v['answer']}")

    print("\n--- TOOL-AUGMENTED, first run ---")
    a1 = run_agent(q)
    print(f"{a1['answer']}\n({a1['latency_sec']}s, {a1['total_tokens']} tokens, "
          f"tools: {len(a1['tools_used'])})")

    print("\n--- TOOL-AUGMENTED, repeat (cache) ---")
    a2 = run_agent(q, verbose=False)
    print(f"{a2['answer']}\n({a2['latency_sec']}s)")

    rows.append({"question": q,
                 "vanilla_answer": v["answer"], "vanilla_sec": v["latency_sec"],
                 "vanilla_tokens": v["total_tokens"],
                 "tools_used": "; ".join(a1["tools_used"]) or "none (answered directly)",
                 "agent_answer": a1["answer"], "agent_sec_first": a1["latency_sec"],
                 "agent_tokens": a1["total_tokens"], "agent_sec_cached": a2["latency_sec"]})

pd.DataFrame(rows).to_csv("tool_app_results.csv", index=False)
print("\nSaved to tool_app_results.csv")
print("Weather tool stats:", stats)


QUESTION: What's the weather in Paris right now? Give the temperature in celsius.

--- VANILLA LLM (3.32s, 58 tokens) ---
I'm unable to provide real-time weather updates. To find the current weather in Paris, I recommend checking a reliable weather website or app for the most accurate and up-to-date information.

--- TOOL-AUGMENTED, first run ---
   -> tool call: get_weather({'city': 'Paris', 'unit': 'celsius'}) = Paris, France: 13.6°C, overcast, humidity 86%, wind 10.7 km/h
The current weather in Paris is 13.6°C, overcast, with a humidity of 86% and a wind speed of 10.7 km/h.
(3.97s, 409 tokens, tools: 1)

--- TOOL-AUGMENTED, repeat (cache) ---
The current weather in Paris is 13.6°C, overcast, with a humidity of 86% and a wind speed of 10.7 km/h.
(0.02s)

QUESTION: What's 42 x 19?

--- VANILLA LLM (1.48s, 22 tokens) ---
42 x 19 equals 798.

--- TOOL-AUGMENTED, first run ---
   -> tool call: calculator({'expression': '42 * 19'}) = 798
42 x 19 equals 798.
(1.96s, 339 tokens, tools: 1)


## Cell 7: Summary table

In [7]:
summary = pd.DataFrame(rows)[["question", "tools_used", "vanilla_sec",
                              "agent_sec_first", "agent_sec_cached"]]
summary

,question,tools_used,vanilla_sec,agent_sec_first,agent_sec_cached
0,What's the weather in Paris right now? Give th...,"get_weather({'city': 'Paris', 'unit': 'celsius'})",3.32,3.97,0.02
1,What's 42 x 19?,calculator({'expression': '42 * 19'}),1.48,1.96,0.01
2,What's the weather in Paris and what is 42 x 1...,get_weather({'city': 'Paris'}); calculator({'e...,0.84,2.01,0.00
3,What is the capital of France?,none (answered directly),0.73,0.91,0.00
